# 04 · Sentiment vs. Trader Performance
> Core relationship analysis — ANOVA, correlation, grouped visuals

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent / "src"))

import pandas as pd
import plotly.express as px
from preprocessing import PROCESSED_DIR  # noqa
from analysis import pnl_by_regime, run_anova, pearson_spearman
from visualisation import (
    plot_pnl_by_regime, plot_winrate_by_regime, plot_correlation_heatmap
)

PROCESSED_DIR = pathlib.Path().resolve().parent / "data" / "processed"
merged = pd.read_csv(PROCESSED_DIR / "merged_dataset.csv")
print(merged.shape)


## 1. PnL Statistics by Regime

In [ ]:
regime_stats = pnl_by_regime(merged)
regime_stats


## 2. ANOVA — Does Regime Significantly Affect PnL?

In [ ]:
anova = run_anova(merged)
print(f"F = {anova['F_statistic']}  |  p = {anova['p_value']}")
print(f"→ {anova['interpretation']}")


## 3. Pearson & Spearman Correlations

In [ ]:
corr = pearson_spearman(merged.dropna(subset=["regime_numeric", "closedpnl"]),
                         "regime_numeric", "closedpnl")
print("Sentiment ↔ PnL Correlation:")
for k, v in corr.items():
    print(f"  {k}: {v}")


## 4. Violin Plot — PnL by Regime

In [ ]:
fig = plot_pnl_by_regime(merged, save=True)
fig.show()


## 5. Win Rate by Regime

In [ ]:
fig = plot_winrate_by_regime(regime_stats, save=True)
fig.show()


## 6. Correlation Heatmap

In [ ]:
import matplotlib.pyplot as plt
fig = plot_correlation_heatmap(merged, save=True)
plt.show()


## 7. Long vs. Short PnL by Regime

In [ ]:
if "is_long" in merged.columns:
    side_regime = merged.groupby(["regime", "is_long"])["closedpnl"].mean().reset_index()
    side_regime["direction"] = side_regime["is_long"].map({1: "Long", 0: "Short"})
    px.bar(side_regime, x="regime", y="closedpnl", color="direction",
           barmode="group", template="plotly_dark",
           title="Mean PnL: Long vs Short by Regime").show()


## ✅ Analysis complete → `05_pattern_discovery.ipynb`